# Notebook 04 — Feature Engineering and Integration

Handles: Merge all datasets, Final Feature Engineering, Validation, Call/Put separation,
Chronological 80/20 split, Final CSV generation.

**Inputs:**
- `data/processed/intermediate/options_cleaned.csv`
- `data/processed/intermediate/vix_cleaned.csv`
- `data/processed/intermediate/yield_curve_cleaned.csv`

**Outputs:**
- `data/processed/final/train/call_train.csv`
- `data/processed/final/train/put_train.csv`
- `data/processed/final/test/call_test.csv`
- `data/processed/final/test/put_test.csv`
- `outputs/reports/data_quality_report.txt`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src', 'preprocessing'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

## Step 1 — Run the Full Integration Pipeline

In [ ]:
import feature_engineering

quality = feature_engineering.run(verbose=True)


## Step 2 — Load and Verify Final Output Files

In [ ]:
base = os.path.join('..', 'data', 'processed', 'final')

call_train = pd.read_csv(os.path.join(base, 'train', 'call_train.csv'), parse_dates=['date', 'expiration'])
put_train  = pd.read_csv(os.path.join(base, 'train', 'put_train.csv'),  parse_dates=['date', 'expiration'])
call_test  = pd.read_csv(os.path.join(base, 'test',  'call_test.csv'),  parse_dates=['date', 'expiration'])
put_test   = pd.read_csv(os.path.join(base, 'test',  'put_test.csv'),   parse_dates=['date', 'expiration'])

print('call_train:', call_train.shape)
print('put_train :', put_train.shape)
print('call_test :', call_test.shape)
print('put_test  :', put_test.shape)


In [ ]:
print('call_train columns:', call_train.columns.tolist())
call_train.head(5)


## Step 3 — Final Schema Validation

In [ ]:
REQUIRED_COLUMNS = [
    'date', 'expiration', 'option_type',
    'S', 'strike', 'bid', 'ask', 'market_price',
    'volume', 'open_interest',
    'days_to_expiration', 'T',
    'moneyness', 'moneyness_category',
    'sigma', 'r', 'market_price_over_K'
]

for name, df in [('call_train', call_train), ('put_train', put_train),
                  ('call_test', call_test), ('put_test', put_test)]:
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    null_counts = df[REQUIRED_COLUMNS].isnull().sum().sum()
    print(f'{name}: missing_cols={missing_cols}, total_nulls={null_counts}')


## Step 4 — Chronological Split Verification

Confirm the train data precedes test data in time (no data leakage).

In [ ]:
for name_train, name_test, df_train, df_test in [
    ('call_train', 'call_test', call_train, call_test),
    ('put_train',  'put_test',  put_train,  put_test)
]:
    train_max = df_train['date'].max()
    test_min  = df_test['date'].min()
    ok = train_max <= test_min
    print(f'{name_train} max date: {train_max.date()}  |  '
          f'{name_test} min date: {test_min.date()}  |  '
          f'Chronological: {"PASS" if ok else "FAIL"}')


## Step 5 — Visualise Final Dataset

In [ ]:
all_data = pd.concat([call_train, put_train, call_test, put_test], ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Final Integrated Dataset — Feature Distributions', fontsize=14)

all_data['market_price'].hist(bins=50, ax=axes[0,0])
axes[0,0].set_title('market_price')

all_data['T'].hist(bins=40, ax=axes[0,1])
axes[0,1].set_title('T (time to expiration in years)')

all_data['moneyness'].hist(bins=50, ax=axes[0,2])
axes[0,2].set_title('moneyness (S/K)')

all_data['sigma'].hist(bins=30, ax=axes[1,0])
axes[1,0].set_title('sigma (prev-day VIX)')

all_data['r'].hist(bins=30, ax=axes[1,1])
axes[1,1].set_title('r (risk-free rate)')

all_data['moneyness_category'].value_counts().plot(kind='bar', ax=axes[1,2])
axes[1,2].set_title('moneyness_category')

plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'figures', '04_final_distributions.png'), dpi=100)
plt.show()
print('Plot saved.')


## Step 6 — Data Quality Report

In [ ]:
report_path = os.path.join('..', 'outputs', 'reports', 'data_quality_report.txt')
with open(report_path, 'r', encoding='utf-8') as f:
    print(f.read())


## Step 7 — Handoff Confirmation

The next team member can load these files and immediately start Black-Scholes calculation:

```python
import pandas as pd

call_train = pd.read_csv('data/processed/final/train/call_train.csv')
put_train  = pd.read_csv('data/processed/final/train/put_train.csv')
call_test  = pd.read_csv('data/processed/final/test/call_test.csv')
put_test   = pd.read_csv('data/processed/final/test/put_test.csv')

# Black-Scholes inputs are ready:
# S = underlying price
# K = strike
# sigma = previous day VIX close (volatility input)
# r = maturity-interpolated risk-free rate (decimal)
# T = time to expiration in years
```

**PREPROCESSING MODULE COMPLETE.**